# Machine Learning - Assignment 01
Abdullah Zafar | BCS233188

## Question 1 - ML FLOW

### Flow of a Machine Learning Model

The Machine Learning flow consists of the following steps:

1. **Problem Statement:** Define the objective clearly (e.g., "Predict the selling price of a house").
2. **Data Collection:** Gather raw data containing relevant attributes such as house size, location, bedrooms, age, and sale price.
3. **Data Preprocessing:** Clean the data, convert data types, and handle missing values.
4. **Features Selection:** Select the most relevant and informative attributes to feed into the model.
5. **Choose & Train Model:** Pick an appropriate algorithm and train it so it learns how the input features relate to the target price.
6. **Cross-Validation & Goal Check:** Check whether predicted prices are close to actual prices for houses not used in training, and evaluate if the training goal is met (e.g., error below a target threshold).
7. **Parameter Tuning:** If the goal is not met, adjust model settings (such as tree depth) and train again.
8. **Deployment:** Integrate the trained model so users can enter unseen house details and receive an estimated price.

### Code Cells

In [44]:
## Download and Load Dataset

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Download and load the Ames House Prices dataset
housing = fetch_openml(data_id=42165, as_frame=True)
data = housing.data.copy()
target = pd.to_numeric(housing.target, errors="coerce")

print(f"Dataset loaded successfully. Total records: {len(data)}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Dataset loaded successfully. Total records: 1460


In [45]:
# Feature Engineering: Create 'HouseAge' from YrSold and YearBuilt
data["HouseAge"] = (
    pd.to_numeric(data["YrSold"], errors="coerce") 
    - pd.to_numeric(data["YearBuilt"], errors="coerce")
)

# Feature Selection: Select the 6 important features
FEATURES = ["GrLivArea", "LotArea", "BedroomAbvGr", "FullBath", "HouseAge", "OverallQual"]
X = data[FEATURES].apply(pd.to_numeric, errors="coerce")

# Keep rows with valid target values
valid_rows = target.notna()
X = X.loc[valid_rows]
y = target.loc[valid_rows]

# Split into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Selected Features: {FEATURES}")
print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

Selected Features: ['GrLivArea', 'LotArea', 'BedroomAbvGr', 'FullBath', 'HouseAge', 'OverallQual']
Training samples: 1168 | Testing samples: 292


In [46]:
# Create Pipeline with median imputation for missing values and Random Forest Regressor
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Parameter tuning using 5-fold Cross-Validation
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 15]
}

grid_search = GridSearchCV(pipeline, param_grid, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best Parameters Found:", grid_search.best_params_)

Best Parameters Found: {'model__max_depth': None, 'model__n_estimators': 100}


In [47]:
# Validate model on unseen test split
test_predictions = best_model.predict(X_test)

print(f"Mean Absolute Error (MAE): ${mean_absolute_error(y_test, test_predictions):,.2f}")
print(f"Root Mean Squared Error (RMSE): ${root_mean_squared_error(y_test, test_predictions):,.2f}")
print(f"R-squared (R2 Score): {r2_score(y_test, test_predictions):.3f}\n")

# Predict on the unseen sample from Week 01 Lecture Slide 6: [2000, 3000, 3, 2, 12, 8]
unseen_house = pd.DataFrame([[2000, 3000, 3, 2, 12, 8]], columns=FEATURES)
predicted_price = best_model.predict(unseen_house)[0]

print(f"Unseen Sample Input: {unseen_house.values[0]}")
print(f"Predicted House Price: ${predicted_price:,.2f}")

Mean Absolute Error (MAE): $20,727.04
Root Mean Squared Error (RMSE): $30,180.97
R-squared (R2 Score): 0.881

Unseen Sample Input: [2000 3000    3    2   12    8]
Predicted House Price: $250,931.61


## Question 2 - Algorithms

### Why Find-S is Not Useful Compared to Candidate Elimination Method

The **Find-S algorithm** has major limitations:
* **Ignores Negative Examples:** Find-S only generalizes from positive examples (`label = 1`) and completely ignores negative examples (`label = 0`), meaning it cannot use negative data to rule out overly broad hypotheses.
* **Outputs Only One Rule:** Many different hypotheses may fit the same training data, but Find-S returns only a single maximally specific hypothesis.
* **No Version Space Tracking:** Unlike Candidate Elimination, Find-S does not maintain the full **Version Space** (all consistent rules bounded between the Specific boundary $S$ and General boundary $G$).
* **Noise Sensitivity:** It assumes the dataset has zero mistakes or noise and cannot detect when contradictory examples make the hypothesis space inconsistent.

### Code Cells

In [48]:
import itertools

# Extended Loan Approval Problem (from Week 2 & 3 slides) with 5 attributes:
# Attributes: [Income, Credit, Job, Default, Collateral]
# Target Concept Rule: Approve (1) if Credit == 'Good' AND Job == 'Stable' AND Default == 'No', else Reject (0)

incomes = ['High', 'Medium', 'Low']
credits = ['Good', 'Poor']
jobs = ['Stable', 'Unstable']
defaults = ['No', 'Yes']
collaterals = ['Yes', 'No']

# Generate all 48 unique combinations + 2 additional samples to make 50 samples
all_combinations = list(itertools.product(incomes, credits, jobs, defaults, collaterals))
raw_samples = all_combinations + [all_combinations[0], all_combinations[1]]

training_data_50 = []
for sample in raw_samples:
    features = list(sample)
    # Apply target concept rule
    label = 1 if (features[1] == 'Good' and features[2] == 'Stable' and features[3] == 'No') else 0
    training_data_50.append((features, label))

print(f"Total samples created: {len(training_data_50)}")
print(f"Number of attributes per sample: {len(training_data_50[0][0])}")
print("First 3 samples:", training_data_50[:3])

Total samples created: 50
Number of attributes per sample: 5
First 3 samples: [(['High', 'Good', 'Stable', 'No', 'Yes'], 1), (['High', 'Good', 'Stable', 'No', 'No'], 1), (['High', 'Good', 'Stable', 'Yes', 'Yes'], 0)]


In [49]:
def is_consistent(hypothesis, features):
    """Check if a hypothesis accepts a given feature vector."""
    for h_val, f_val in zip(hypothesis, features):
        if h_val == 'Ø':
            return False
        if h_val != '?' and h_val != f_val:
            return False
    return True

def is_more_general(h1, h2):
    """Return True if h1 is more general than or equal to h2."""
    for v1, v2 in zip(h1, h2):
        if v1 != '?' and (v1 != v2 and v2 != 'Ø'):
            return False
    return True

def candidate_elimination(training_data, domains):
    num_attr = len(domains)
    # Initialize S to most specific and G to most general
    S = [['Ø'] * num_attr]
    G = [['?'] * num_attr]

    for features, label in training_data:
        if label == 1:  # Positive example
            # 1. Remove inconsistent hypotheses from G
            G = [g for g in G if is_consistent(g, features)]
            
            # 2. Minimally generalize S to cover the positive example
            new_S = []
            for s in S:
                s_copy = s.copy()
                for i in range(num_attr):
                    if s_copy[i] == 'Ø':
                        s_copy[i] = features[i]
                    elif s_copy[i] != features[i]:
                        s_copy[i] = '?'
                new_S.append(s_copy)
            
            # Keep only hypotheses in S that are more specific than some hypothesis in G
            S = [s for s in new_S if any(is_more_general(g, s) for g in G)]

        else:  # Negative example (label == 0)
            # 1. Remove inconsistent hypotheses from S
            S = [s for s in S if not is_consistent(s, features)]
            
            # 2. Minimally specialize G so they reject the negative example
            new_G = []
            for g in G:
                if is_consistent(g, features):
                    for i in range(num_attr):
                        if g[i] == '?':
                            for val in domains[i]:
                                if val != features[i]:
                                    g_spec = g.copy()
                                    g_spec[i] = val
                                    # Keep specialization only if it is more general than some member of S
                                    if any(is_more_general(g_spec, s) for s in S):
                                        if g_spec not in new_G:
                                            new_G.append(g_spec)
                else:
                    new_G.append(g)
            
            # 3. Remove redundant (more specific) hypotheses in G
            G = [g1 for g1 in new_G if not any(g1 != g2 and is_more_general(g2, g1) for g2 in new_G)]

    return S, G

# Define attribute domains and run the algorithm
domains = [incomes, credits, jobs, defaults, collaterals]
S_final, G_final = candidate_elimination(training_data_50, domains)

print("Final Specific Boundary (S):", S_final)
print("Final General Boundary (G):", G_final)

Final Specific Boundary (S): [['?', 'Good', 'Stable', 'No', '?']]
Final General Boundary (G): [['?', 'Good', 'Stable', 'No', '?']]


### Convergence and Comparison Explanation

* **Is the model converged or not?**
  **Yes, the model has converged.** As stated in the Week 3 slides, convergence occurs when the Specific boundary ($S$) and the General boundary ($G$) move closer and become identical (`S == G = [['?', 'Good', 'Stable', 'No', '?']]`). Because $S$ and $G$ are identical, the version space shrinks to a single consistent hypothesis that perfectly represents the target concept.

* **Why is Candidate Elimination a better choice than Find-S for your data?**
  1. **Uses Both Positive and Negative Examples:** Find-S ignores all negative loan applications (`label = 0`), whereas Candidate Elimination actively uses negative samples to specialize $G$ and eliminate invalid rules (such as rules that ignore credit status or job stability).
  2. **Maintains the Entire Version Space:** Instead of guessing a single specific rule like Find-S, Candidate Elimination tracks all consistent hypotheses between boundaries $S$ and $G$, proving with certainty when the true target rule is found.
  3. **Detects Noise/Contradictions:** If our 50 samples contained a mislabeled or contradictory example, Candidate Elimination would detect it by collapsing to an empty version space, whereas Find-S would blindly over-generalize.